# Classical Model Prediction Pipeline

This notebook uses the trained classical ML models to predict **MaxTemp** and **RainTomorrow**
for a given date, producing the same CSV format as the deep-learning path.

Target CSV columns (one row per city):
```
Ville,Date_Predite,Temp_Predite,Prob_Pluie,Pluie_Prevue
```

## Cell 1 — Imports and config

In [ ]:
import pandas as pd
import numpy as np
import joblib
from classical_models import FEATURE_COLUMNS, TARGET_RAIN

DATA_PATH = "data/clean_data.csv"
DATE = "2017-03-18"          # change as needed

# Feature columns for rain classification (full set)
RAIN_FEATURES = FEATURE_COLUMNS

# Feature columns for temp regression (MaxTemp excluded to avoid leakage)
TEMP_FEATURES = [c for c in FEATURE_COLUMNS if c != "MaxTemp"]

## Cell 2 — Load models

In [ ]:
rain_model = joblib.load("saved_models/xgboost.joblib")        # best classical rain model
temp_model = joblib.load("saved_models/temp_regressor.joblib") # temperature regressor

## Cell 3 — Prediction function

In [ ]:
def predict_for_specific_day_classical(
    rain_model,
    temp_model,
    df: pd.DataFrame,
    target_date: str,
) -> pd.DataFrame:
    """
    For every city that has a row on `target_date`, predict:
      - Temp_Predite  : MaxTemp (°C) via regression model
      - Prob_Pluie    : P(RainTomorrow=1) via rain classifier
      - Pluie_Prevue  : 1 if Prob_Pluie > 0.5 else 0

    Classical models use a single tabular row per city (no sequence window).
    The row used is the one matching target_date itself.

    Returns a DataFrame with columns:
        Ville, Date_Predite, Temp_Predite, Prob_Pluie, Pluie_Prevue
    """
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    target_dt = pd.to_datetime(target_date)

    day_df = df[df["Date"] == target_dt].copy()

    if day_df.empty:
        raise ValueError(f"No data found for date {target_date}")

    # ── Rain prediction ────────────────────────────────────────────────────────────────
    # Drop rows where any rain feature is NaN
    rain_df = day_df.dropna(subset=RAIN_FEATURES)
    X_rain = rain_df[RAIN_FEATURES]
    prob_rain = rain_model.predict_proba(X_rain)[:, 1]

    # ── Temperature prediction ───────────────────────────────────────────────────────────
    temp_df = day_df.dropna(subset=TEMP_FEATURES)
    X_temp = temp_df[TEMP_FEATURES]
    pred_temp = temp_model.predict(X_temp)

    # ── Merge on city (Location) ───────────────────────────────────────────────────────
    rain_result = pd.DataFrame({
        "Location": rain_df["Location"].values,
        "Prob_Pluie": prob_rain,
    })
    temp_result = pd.DataFrame({
        "Location": temp_df["Location"].values,
        "Temp_Predite": pred_temp,
    })

    merged = rain_result.merge(temp_result, on="Location", how="inner")

    output = pd.DataFrame({
        "Ville":        merged["Location"],
        "Date_Predite": target_date,
        "Temp_Predite": merged["Temp_Predite"].round(2),
        "Prob_Pluie":   merged["Prob_Pluie"].round(4),
        "Pluie_Prevue": (merged["Prob_Pluie"] > 0.5).astype(int),
    })

    output.to_csv(f"predict/predict_{target_date}.csv", index=False)
    print(f"Saved {len(output)} rows \u2192 predict/predict_{target_date}.csv")
    return output

## Cell 4 — Run prediction

In [ ]:
df = pd.read_csv(DATA_PATH)
results = predict_for_specific_day_classical(rain_model, temp_model, df, DATE)
results

## Cell 5 — Evaluation against actuals

In [ ]:
def evaluate_classical_performance(
    rain_model,
    temp_model,
    df: pd.DataFrame,
    target_date: str,
) -> tuple[float, float]:
    """
    Compare predictions against actuals for target_date.
    Returns (mean_absolute_temp_error, rain_accuracy).
    """
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    target_dt = pd.to_datetime(target_date)
    day_df = df[df["Date"] == target_dt].dropna(subset=RAIN_FEATURES + ["MaxTemp", TARGET_RAIN])

    X_rain = day_df[RAIN_FEATURES]
    X_temp = day_df[TEMP_FEATURES]

    prob_rain   = rain_model.predict_proba(X_rain)[:, 1]
    pred_labels = (prob_rain > 0.5).astype(int)
    pred_temp   = temp_model.predict(X_temp)

    actual_temp = day_df["MaxTemp"].values
    actual_rain = day_df[TARGET_RAIN].values   # already binary (0/1)

    mae_temp    = float(np.mean(np.abs(pred_temp - actual_temp)))
    rain_acc    = float(np.mean(pred_labels == actual_rain))

    print(f"--- Performances au {target_date} (classical models) ---")
    print(f"\u00c9cart de temp\u00e9rature moyen : {mae_temp:.2f}\u00b0C")
    print(f"Accuracy Pluie             : {rain_acc * 100:.1f}%")
    return mae_temp, rain_acc


mae, acc = evaluate_classical_performance(rain_model, temp_model, df, DATE)

## Sanity Checks

| Check | Expected |
|-------|----------|
| `results.columns.tolist()` | `['Ville', 'Date_Predite', 'Temp_Predite', 'Prob_Pluie', 'Pluie_Prevue']` |
| `results["Temp_Predite"].between(5, 50).all()` | `True` (no negative temps) |
| `results["Prob_Pluie"].between(0, 1).all()` | `True` |
| `results["Pluie_Prevue"].isin([0, 1]).all()` | `True` |
| CSV file exists at `predict/predict_{DATE}.csv` | `True` |

In [ ]:
import os

print("Columns:", results.columns.tolist())
print("Expected: ['Ville', 'Date_Predite', 'Temp_Predite', 'Prob_Pluie', 'Pluie_Prevue']")
print()
print(f"Temp in [5, 50]: {results['Temp_Predite'].between(5, 50).all()}")
print(f"Prob in [0, 1]:  {results['Prob_Pluie'].between(0, 1).all()}")
print(f"Rain in {{0,1}}: {results['Pluie_Prevue'].isin([0, 1]).all()}")
print(f"CSV exists:      {os.path.exists(f'predict/predict_{DATE}.csv')}")